In [1]:
# ============================
# Cell 1 — Setup & Data Loader
# ============================

# Dash / components
from dash import Dash, dcc, html, dash_table as dt
from dash.dependencies import Input, Output, State
import dash_leaflet as dl

# Plotting / visualization
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math, hashlib

# System utilities
import base64, os, logging

# Database (MongoDB + CRUD helper module)
from animal_shelter import AnimalShelter   # ensure this matches your file/class
import pymongo

# .env support
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# -------- Logging --------
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s [%(name)s] %(message)s")
logger = logging.getLogger("dashboard")

# -------- Config --------
APP_TITLE = "SNHU CS-340 Dashboard"       # used later in layout
CSV_FALLBACK_PATH = "animals.csv"         # local fallback CSV

# Mongo connection 
USER = os.getenv("MONGO_USER", "")
PASS = os.getenv("MONGO_PASS", "")
HOST = os.getenv("MONGO_HOST", "localhost")
PORT = int(os.getenv("MONGO_PORT", "27017"))
DB   = os.getenv("MONGO_DB", "AAC")
COL  = os.getenv("MONGO_COL", "animals")

# Default read options 
READ_QUERY = {}
READ_PROJECTION = None   # e.g., {"name":1,"breed":1,"age":1}
READ_LIMIT = None        # e.g., 100

# Shared per-session cache
_DATA_CACHE = {"df": None, "source": None}

def load_data_once():
    """
    Try Mongo first; on error or empty, fall back to CSV.
    Returns (df, source_str) where source_str in {"MongoDB","CSV"}.
    """
    if _DATA_CACHE["df"] is not None:
        return _DATA_CACHE["df"], _DATA_CACHE["source"]

    # Connect via CRUD helper
    shelter = AnimalShelter(USER, PASS, HOST, PORT, DB, COL)

    # Try Mongo
    try:
        logger.info("Attempting Mongo read...")
        docs = shelter.read(READ_QUERY, projection=READ_PROJECTION, limit=READ_LIMIT)
        df = pd.DataFrame.from_records(docs)
        if df.empty:
            raise ValueError("Mongo returned 0 rows")
        # Clean up ObjectId if present to avoid dash_table issues
        df.drop(columns=["_id"], inplace=True, errors="ignore")
        _DATA_CACHE["df"], _DATA_CACHE["source"] = df, "MongoDB"
        logger.info("Loaded %d rows from MongoDB", len(df))
        return df, "MongoDB"
    except Exception as e:
        logger.warning("Mongo unavailable or empty (%s). Falling back to CSV: %s", type(e).__name__, e)
        if not os.path.exists(CSV_FALLBACK_PATH):
            logger.error("CSV fallback not found at %s", CSV_FALLBACK_PATH)
            raise
        df = pd.read_csv(CSV_FALLBACK_PATH)
        # Keep schema friendly for Dash table
        df.drop(columns=["_id"], inplace=True, errors="ignore")
        _DATA_CACHE["df"], _DATA_CACHE["source"] = df, "CSV"
        logger.info("Loaded %d rows from CSV", len(df))
        return df, "CSV"

# --- Load once so downstream cells can use df and SOURCE immediately ---
df, SOURCE = load_data_once()


2025-09-30 14:13:45,165 INFO [animal_shelter] Connecting to MongoDB: localhost/AAC (timeout=5s)
2025-09-30 14:13:45,327 INFO [dashboard] Attempting Mongo read...
2025-09-30 14:13:46,661 INFO [animal_shelter] read: query={} projection=None limit=None -> 173775 valid docs (1212.0 ms)
2025-09-30 14:13:47,005 INFO [dashboard] Loaded 173775 rows from MongoDB


In [2]:
# ============================
# Cell 2 — Layout + Callbacks 
# ============================

import time

app = Dash(__name__)

# --- Logo ---
encoded_image = None
logo_path = "Grazioso Salvare Logo.png"
if os.path.exists(logo_path):
    with open(logo_path, "rb") as f:
        encoded_image = base64.b64encode(f.read()).decode("utf-8")

status_text = f"Data source: {SOURCE}"

# ---------- Chart helper ----------
def _default_figure_for(df_in):
    # Prefer a known categorical
    if "Breed" in df_in.columns and df_in["Breed"].notna().any():
        return px.histogram(df_in, x="Breed", title="Distribution of Breed")

    # Else pick a small-cardinality categorical column
    cat_cols = [c for c in df_in.columns
                if df_in[c].dtype == "object" and df_in[c].nunique() <= 50 and df_in[c].notna().any()]
    if cat_cols:
        col = cat_cols[0]
        return px.histogram(df_in, x=col, title=f"Distribution of {col}")

    # Else try a simple numeric fallback
    num_cols = [c for c in df_in.columns if pd.api.types.is_numeric_dtype(df_in[c])]
    if num_cols:
        base = df_in.reset_index()
        ycol = num_cols[0]
        return px.scatter(base, x="index", y=ycol, title=f"Sample Plot: {ycol} vs index")

    # Last resort
    return px.scatter(pd.DataFrame({"x": [0], "y": [0]}), x="x", y="y", title="Waiting for data")

# ---------- Vectorized filtering helpers ----------
def _series_false(df_in):
    return pd.Series(False, index=df_in.index) if len(df_in.index) else pd.Series([], dtype=bool)

def _eq(df_in, col, val):
    return (df_in[col] == val) if col in df_in.columns else _series_false(df_in)

def _isin(df_in, col, values):
    return (df_in[col].isin(values)) if col in df_in.columns else _series_false(df_in)

FILTER_CACHE = {}  # cache by filter_type -> filtered df

def filter_dataframe_by_rescue(df_in, filter_type):
    cached = FILTER_CACHE.get(filter_type)
    if cached is not None:
        return cached

    d = df_in.copy()

    if filter_type == "water":
        breeds = ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]
        mask = _eq(d, "Animal Type", "Dog") & _isin(d, "Breed", breeds) & _eq(d, "Sex upon Outcome", "Intact Female")
        d = d[mask]
    elif filter_type == "mount":
        breeds = ["German Shepard", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]
        mask = _eq(d, "Animal Type", "Dog") & _isin(d, "Breed", breeds) & _eq(d, "Sex upon Outcome", "Intact Male")
        d = d[mask]
    elif filter_type == "disaster":
        breeds = ["Doberman Pinscher", "German Shepard", "Golden Retriever", "Bloodhound", "Rottweiler"]
        mask = _eq(d, "Animal Type", "Dog") & _isin(d, "Breed", breeds) & _eq(d, "Sex upon Outcome", "Intact Male")
        d = d[mask]
    else:
        # reset -> no filter
        pass

    d.drop(columns=["_id"], inplace=True, errors="ignore")
    FILTER_CACHE[filter_type] = d
    return d

# ---------- Vectorized Lat/Lon synthesis ----------
AUSTIN_CENTER = [30.2672, -97.7431]
RADIUS_KM = 5.0

def ensure_latlon(df_in):
    d = df_in.copy()
    if "Latitude" in d.columns and "Longitude" in d.columns:
        return d

    # Deterministic seeds from Animal ID (or index)
    key_series = d["Animal ID"].astype(str) if "Animal ID" in d.columns else d.index.astype(str)
    # numeric seeds (32-bit)
    seeds = key_series.apply(lambda k: int(hashlib.sha256(k.encode()).hexdigest(), 16) % (2**32))

    # Generate uniform offsets per row
    # Using apply here avoids Python loops over rows and is reasonably efficient for notebook scale
    rngs = seeds.apply(np.random.RandomState)
    dlat = (RADIUS_KM / 111.0) * rngs.apply(lambda r: r.uniform(-1, 1))
    dlon = (RADIUS_KM / (111.0 * math.cos(math.radians(AUSTIN_CENTER[0])))) * rngs.apply(lambda r: r.uniform(-1, 1))

    d["Latitude"] = AUSTIN_CENTER[0] + dlat.values
    d["Longitude"] = AUSTIN_CENTER[1] + dlon.values
    return d

# ---------- Layout ----------
app.layout = html.Div(
    className="app-container",
    children=[
        html.Div(id="hidden-div", style={"display": "none"}),

        html.H1(APP_TITLE, className="app-title"),
        html.H2("Steven Copeland-Helzer • Project 2 Dashboard", className="app-subtitle"),

        html.Div(status_text, id="status-banner", className="status-banner", role="status", **{"aria-live": "polite"}),

        # Logo
        html.Div(
            children=html.Img(
                id="customer-image",
                src=f"data:image/png;base64,{encoded_image}" if encoded_image else None,
                alt="Grazioso Salvare logo"
            ),
            className="card"
        ) if encoded_image else html.Div(),

        # Filter controls
        html.Div(
            className="card",
            children=[
                dcc.RadioItems(
                    id="filter-type",
                    options=[
                        {"label": "Water Rescue", "value": "water"},
                        {"label": "Mountain/Wilderness Rescue", "value": "mount"},
                        {"label": "Disaster Rescue and Individual Tracking", "value": "disaster"},
                        {"label": "Reset", "value": "reset"},
                    ],
                    value="reset",
                )
            ],
            **{"aria-label": "Rescue type filter"}
        ),

        # Data table
        html.Div(
            className="card table-container",
            children=[
                dt.DataTable(
                    id="datatable-id",
                    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                    data=df.to_dict("records"),
                    editable=False,
                    filter_action="native",
                    sort_action="native",
                    sort_mode="multi",
                    column_selectable="single",
                    row_selectable="single",
                    row_deletable=False,
                    selected_columns=[],
                    selected_rows=[0],
                    page_action="native",
                    page_current=0,
                    page_size=10,
                )
            ],
            **{"aria-label": "Animals data table"}
        ),

        # Graph + Map
        html.Div(
            className="row-flex",
            children=[
                html.Div(
                    id="graph-id",
                    className="card col-6",
                    children=[dcc.Graph(figure=_default_figure_for(df))]
                ),
                dl.Map(
                    id="map-canvas",
                    className="card map-canvas",
                    center=AUSTIN_CENTER,
                    zoom=10,
                    children=[dl.TileLayer(id="base-layer-id")]
                ),
            ],
        ),
    ],
)

# ============================
# Callbacks
# ============================

# Radio filter -> DataTable 
@app.callback(
    [Output("datatable-id", "data"),
     Output("datatable-id", "columns")],
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    t0 = time.perf_counter()
    dfl = filter_dataframe_by_rescue(df, filter_type)
    cols = [{"name": c, "id": c, "deletable": False, "selectable": True} for c in dfl.columns]
    dt_ms = (time.perf_counter() - t0) * 1000
    logger.info("filter=%s -> %d rows (%.1f ms)", filter_type, len(dfl), dt_ms)
    return dfl.to_dict("records"), cols

# DataTable -> Graph 
from dash.exceptions import PreventUpdate

@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "data"),
     Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(table_data, view_data):
    use_data = view_data if view_data else table_data
    if not use_data:
        raise PreventUpdate
    dff = pd.DataFrame(use_data)

    if "Breed" in dff.columns and dff["Breed"].notna().any():
        return dcc.Graph(figure=px.histogram(dff, x="Breed", title="Distribution of Breed"))

    cat_cols = [c for c in dff.columns
                if dff[c].dtype == "object" and dff[c].nunique() <= 50 and dff[c].notna().any()]
    if cat_cols:
        col = cat_cols[0]
        return dcc.Graph(figure=px.histogram(dff, x=col, title=f"Distribution of {col}"))

    num_cols = [c for c in dff.columns if pd.api.types.is_numeric_dtype(dff[c])]
    if num_cols:
        base = dff.reset_index()
        ycol = num_cols[0]
        return dcc.Graph(figure=px.scatter(base, x="index", y=ycol, title=f"Sample Plot: {ycol} vs index"))

    return html.Div("No plottable columns found in the current view.")

# DataTable -> Map 
@app.callback(
    Output("map-canvas", "center"),
    Output("map-canvas", "zoom"),
    Output("map-canvas", "children"),
    [Input("datatable-id", "derived_viewport_data"),
     Input("datatable-id", "derived_viewport_selected_rows")]
)
def update_map(viewData, selected_rows):
    base = [dl.TileLayer(id="base-layer-id")]
    if not viewData:
        return AUSTIN_CENTER, 10, base

    dff = pd.DataFrame(viewData)
    dff = ensure_latlon(dff)

    idx = (selected_rows or [0])[-1]
    idx = max(0, min(idx, len(dff) - 1))

    try:
        lat = float(dff.loc[idx, "Latitude"])
        lon = float(dff.loc[idx, "Longitude"])
    except Exception:
        return AUSTIN_CENTER, 10, base

    center = [lat, lon]
    marker = dl.Marker(position=center, children=[
        dl.Tooltip(str(dff.loc[idx].get("Breed", "Unknown"))),
        dl.Popup([
            html.H4(str(dff.loc[idx].get("Name", "")) or "Animal"),
            html.P(f"{dff.loc[idx].get('Animal Type', '')} • {dff.loc[idx].get('Breed', '')}")
        ])
    ])
    return center, 12, [dl.TileLayer(id="base-layer-id"), marker]

# Highlight selected columns
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in (selected_columns or [])]


In [3]:
app.run(debug=False)

2025-09-30 14:14:20,372 INFO [dashboard] filter=reset -> 173775 rows (112.9 ms)
2025-09-30 14:14:27,215 INFO [dashboard] filter=reset -> 173775 rows (0.0 ms)
2025-09-30 14:14:48,153 INFO [dashboard] filter=disaster -> 113 rows (135.8 ms)
